# Apple Depth Pro - Image Depth Estimation (PoC)

Lightweight proof of concept for monocular metric depth estimation on a single image using [Apple's Depth Pro](https://github.com/apple/ml-depth-pro).

**Recommended runtime:** GPU (T4 is fine). `Runtime -> Change runtime type -> T4 GPU`.

## 1. Install Depth Pro

In [ ]:
!git clone https://github.com/apple/ml-depth-pro.git
%cd ml-depth-pro
!pip install -e . -q

# Depth Pro pins numpy<2; Colab ships numpy 2.x. The pip downgrade above
# leaves already-imported C extensions ABI-mismatched, so we restart the
# kernel here. After the auto-restart, SKIP this cell and run from cell 2.
import os
os.kill(os.getpid(), 9)

## 2. Download pretrained checkpoint (~1.9 GB)

In [ ]:
!source get_pretrained_models.sh

## 3. Load the model

> After the kernel auto-restart from cell 1, start running here. The next cell re-enters the repo dir and adds it to `sys.path`.

In [ ]:
%cd /content/ml-depth-pro
import sys, os
src_path = os.path.abspath('src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import depth_pro

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, transform = depth_pro.create_model_and_transforms(device=device, precision=torch.float16)
model.eval()
print('Loaded on', device)

## 4. Pick an image

Either upload your own or use the sample bundled with the repo.

In [ ]:
IMAGE_PATH = 'data/example.jpg'  # change to your uploaded file path if desired

# To upload your own in Colab, uncomment:
# from google.colab import files
# uploaded = files.upload()
# IMAGE_PATH = next(iter(uploaded))

## 5. Run inference

In [ ]:
image, _, f_px = depth_pro.load_rgb(IMAGE_PATH)
image_t = transform(image)

with torch.no_grad():
    prediction = model.infer(image_t, f_px=f_px)

depth = prediction['depth'].cpu().numpy()           # metric depth in meters
focallength_px = prediction['focallength_px'].item()
print(f'Depth shape: {depth.shape}, min={depth.min():.2f} m, max={depth.max():.2f} m')
print(f'Estimated focal length (px): {focallength_px:.1f}')

## 6. Visualize

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

inv_depth = 1.0 / np.clip(depth, 1e-3, None)
p2, p98 = np.percentile(inv_depth, [2, 98])
inv_depth_vis = np.clip((inv_depth - p2) / (p98 - p2 + 1e-8), 0, 1)

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(image); ax[0].set_title('Input'); ax[0].axis('off')
im = ax[1].imshow(inv_depth_vis, cmap='turbo')
ax[1].set_title('Depth (inverse, normalized)'); ax[1].axis('off')
plt.colorbar(im, ax=ax[1], fraction=0.046)
plt.tight_layout(); plt.show()

## 7. Save raw depth (optional)

In [ ]:
np.save('depth_meters.npy', depth)
print('Saved depth_meters.npy')